In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
import os

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Advanced ML Libraries
import xgboost as xgb
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

# Metrics and Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
    )

# Statistical Libraries
from scipy import stats
from scipy.stats import chi2_contingency

#Progress Bar
from tqdm.auto import tqdm

In [2]:
#Data Loading
class FraudDetectionPipeline:
    def __init__(self):
        self.df = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.models = {}
        self.results = {}
        self.feature_importance = {}

    def load_data(self):
        try:
            train_transaction = pd.read_csv("train_transaction.csv")
            print(f"Transaction data shape:{train_transaction.shape}")
            
            train_identity = pd.read_csv("train_identity.csv")
            print(f"Identity data shape: {train_identity.shape}")

            self.df = train_transaction.merge(train_identity, how="left", on="TransactionID")
            print(f"Merged dataset shape:{self.df.shape}")

            #Basic statistical overview
            print(f"\nDataset overview")
            print(f"Total transaction: {len(self.df):,}")
            print(f"Total features: {self.df.shape[1]}")
            print(f"Overall fraud transaction: {self.df["isFraud"].sum():,}")
            print(f"Fraud Percentage:{self.df["isFraud"].mean():.3%}")

            return True
        
        except Exception as exc:
            print(f"Error loading data:{exc}")

            return False
        
    #Exploratory Data Analysis
    def ExploratoryDataAnalysis(self):
        print("\nDataset Info:")
        print(f"Dataset shape:{self.df.shape}")
        print(f"Data type:{self.df.dtypes.value_counts()}")

        #Null value analysis
        missing_data = self.df.isnull().sum()
        missing_percent = (missing_data / len(self.df)) * 100
        missing_df = pd.DataFrame({
            "MissingCount": missing_data,
            "MissingPercentage": missing_percent
        }).sort_values("MissingPercentage", ascending=False)

        print(f"\nFeature with more than 50% missing value: {len(missing_df[missing_df.MissingPercentage > 50])}")
        print(f"Feature with no missing value: {len(missing_df[missing_df.MissingPercentage == 0])}")

        #Variable Analysis
        fraud_stats = self.df["isFraud"].value_counts()
        print("\nFraud Detection:")
        print(f"notFraud (0): {fraud_stats[0]:,} ({fraud_stats [0] / len(self.df) * 100:.2f}%)")
        print(f"Fraud (1): {fraud_stats[1]:,} ({fraud_stats [1] / len(self.df) * 100:.2f}%)")
        print(f"Ratio: {fraud_stats[0]/fraud_stats[1]:.1f}:1")

        #Transaction Amount Analysis
        

In [3]:
pipeline = FraudDetectionPipeline()
pipeline.load_data()

Transaction data shape:(590540, 394)
Identity data shape: (144233, 41)
Merged dataset shape:(590540, 434)

Dataset overview
Total transaction: 590,540
Total features: 434
Overall fraud transaction: 20,663
Fraud Percentage:3.499%


True

In [4]:
pipeline = FraudDetectionPipeline()
pipeline.load_data()
pipeline.ExploratoryDataAnalysis()

Transaction data shape:(590540, 394)
Identity data shape: (144233, 41)
Merged dataset shape:(590540, 434)

Dataset overview
Total transaction: 590,540
Total features: 434
Overall fraud transaction: 20,663
Fraud Percentage:3.499%

Dataset Info:
Dataset shape:(590540, 434)
Data type:float64    399
str         31
int64        4
Name: count, dtype: int64

Feature with more than 50% missing value: 214
Feature with no missing value: 20

Fraud Detection:
notFraud (0): 569,877 (96.50%)
Fraud (1): 20,663 (3.50%)
Ratio: 27.6:1
